<a href="https://colab.research.google.com/github/joae1234/movie_llm/blob/main/back_end.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from google.colab import userdata
userdata.get('movies_key')

API_KEY = userdata.get('movies_key')
BASE_URL = "https://api.themoviedb.org/3"

def get_movies(page=1, language="pt-BR"):
    url = f"{BASE_URL}/movie/popular?api_key={API_KEY}&language={language}&page={page}"
    response = requests.get(url)
    return response.json()

def get_movie_details(movie_id, language="pt-BR"):
    url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}&language={language}"
    response = requests.get(url)
    return response.json()

# Define the target genres
target_genres = ["Ação", "Comédia", "Drama", "Ficção científica", "Terror"]

# Function to get the single genre based on the target list
def get_single_genre(genres):
    if not genres:
        return None
    for genre in genres:
        if genre["name"] in target_genres:
            return genre["name"]
    return None # Return None if no target genre is found

# Collect movies until 1000 with target genres are found
movies = []
page = 1
max_pages_to_check = 100 # Safeguard against infinite loops

while len([m for m in movies if m['genre'] is not None]) < 1000 and page <= max_pages_to_check:
    data = get_movies(page)
    if not data or not data.get("results"):
        break # Stop if no more data is returned

    for m in data["results"]:
        details = get_movie_details(m["id"])
        genres = details.get("genres", [])
        single_genre = get_single_genre(genres)
        if single_genre in target_genres: # Only add movies with target genres
             movies.append({
                "id": m["id"],
                "title": m["title"],
                "overview": m["overview"],
                "genre": single_genre # Store the single genre
            })

    page += 1
    # Optional: Add a small delay here if needed to avoid hitting API rate limits
    # import time
    # time.sleep(0.1)


df = pd.DataFrame(movies)
display(df.head())
print(f"Collected {len(df)} movies with target genres.")


,id,title,overview,genre
0,507244,Caçadores do Fim do Mundo,Uma década após uma tempestade solar devastar ...,Ficção científica
1,1156594,Nossa Culpa,O casamento de Jenna e Lion marca o tão espera...,Drama
2,755898,A Guerra dos Mundos,Will Radford é um renomado analista de ciberse...,Ficção científica
3,1305717,Hunting Grounds,Uma mãe que foge do marido ligado à máfia enco...,Ação
4,1306525,O Elixir,Um elixir desperta os mortos-vivos em uma vila...,Terror


Collected 1013 movies with target genres.
